# Austrian law RAG – 03: answers with citations (milestone 3)

**Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*. About 15–20 minutes
(most of it downloading the language model once).

For each question: retrieve the best chunks (E5 → reranker), let a language model write a short
German answer that cites §§ like `[AZG § 11]`, then check automatically

- whether the correct § was retrieved at all,
- whether the answer cites it,
- whether every cited § was really in the context (no invented citations),
- whether the answer contains the key fact (e.g. *11 Stunden*) checked against the law text.

Default model: **Qwen2.5-7B-Instruct**, open and free, runs on the T4. To use Claude instead, see the
option cell below.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE – switch the runtime to T4 GPU first')

In [ ]:
import os
if not os.path.exists('/content/austrian-law-rag'):
    !git clone -q https://github.com/zarekasif/austrian-law-rag.git /content/austrian-law-rag
else:
    !git -C /content/austrian-law-rag pull -q
%cd /content/austrian-law-rag
!pip install -q -e ".[dense,generate,api]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/austrian-law-rag'
CORPUS = f'{DATA}/corpus.jsonl'
QUESTIONS = 'data/questions.jsonl'   # the repo version: it has the checked key facts

## Option: which language model

Leave `BACKEND = 'hf'` for the free open model. For Claude: add your API key in Colab's
**Secrets** panel (key icon on the left) as `ANTHROPIC_API_KEY`, then set `BACKEND = 'anthropic'`.
A run over 22 questions costs a few cents.

In [ ]:
BACKEND = 'hf'          # 'hf' (Qwen2.5-7B-Instruct, free) or 'anthropic' (Claude, needs a key)
MODEL = ''              # leave empty for the default of the backend

if BACKEND == 'anthropic':
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
OUT = f'{DATA}/results_m3_{BACKEND}'
MODEL_ARG = f'--model "{MODEL}"' if MODEL else ''

## Generate and score the answers

In [ ]:
!python -m lawrag.eval_answers --corpus "$CORPUS" --questions "$QUESTIONS" --cache "$DATA/embeddings" \
    --out "$OUT" --backend $BACKEND $MODEL_ARG

## Read the answers

`ok` = cites the correct §, `--` = answered but cited something else, `??` = said it cannot answer.
Please read them: the automatic checks catch wrong citations, not every wrong statement.

In [ ]:
import json, textwrap
for line in open(f'{OUT}/answers.jsonl', encoding='utf-8'):
    r = json.loads(line)
    mark = 'ok' if r['cites_gold'] else ('??' if r['abstained'] else '--')
    print(f"{mark} {r['id']}  {r['question']}\n   gold: {r['gold']}  cited: {r['citations']}  fact: {r['fact_hit']}")
    print(textwrap.indent(textwrap.fill(r['answer'], 100), '   '), '\n')

Paste the summary block (the lines with `%`) into the chat. For a careful evaluation, open
`review.csv` in `MyDrive/austrian-law-rag/results_m3_hf/`, mark `manual_correct` as yes/no for
each answer and upload it.